# 🩸 JORINOVA NEXUS — Peripheral Blood Smear (PBS) detector (fine-tuning)

Fine-tunes a **YOLOv8** detector on annotated blood-smear images and produces
`pbs.pt` for the app's vision service. It finds **normal cells + abnormal**
**morphologies**; the app then maps each abnormality to its **related disorders**.

> **Fine-tuning, NOT from scratch** — training starts from pre-trained weights
> (COCO `yolov8s.pt`, or your own blood-domain `malaria.pt`) and adapts them.

**Before you start:** Runtime → **Change runtime type** → **T4 GPU**.
Run each cell top-to-bottom (Shift+Enter).

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Get the project code + mount Drive
Pulls the class map / disorders and the folder the weights go into. Mounting Drive
lets us fine-tune from your `malaria.pt` and keep a backup that survives a runtime reset.

In [ ]:
import os
from getpass import getpass
REPO = 'https://github.com/dujoely1/JORINOVA-NEXUS.git'
tok = getpass('GitHub token (press Enter if the repo is public): ').strip()
url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
os.system(f'rm -rf /content/nexus && git clone --depth 1 {url} /content/nexus')
print('repo cloned:', os.path.exists('/content/nexus/ml/pbs'))
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_pbs', exist_ok=True)
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data — run **Option A** *or* Option B
**Option A needs NO key** (BCCD: RBC/WBC/platelets). Option B adds abnormalities from Roboflow.

### ✅ Option A — BCCD (RBC / WBC / platelets), no key needed
Public VOC dataset → converted to YOLO. Good normal-cell baseline. Class names
(`rbc`,`wbc`,`platelet`) match the app's disorder map.

In [ ]:
import os, urllib.request, zipfile, shutil
import xml.etree.ElementTree as ET
from pathlib import Path
os.makedirs('/content/data', exist_ok=True)
zp = '/content/data/bccd.zip'
if not os.path.exists('/content/data/BCCD_Dataset-master'):
    if not os.path.exists(zp):
        print('downloading BCCD (~7 MB)...')
        urllib.request.urlretrieve('https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip', zp)
    with zipfile.ZipFile(zp) as z: z.extractall('/content/data')
root = Path('/content/data/BCCD_Dataset-master/BCCD')
CLASSES = ['rbc', 'wbc', 'platelet']                 # unified names (match pbs_disorders.json)
VOC2NAME = {'RBC': 'rbc', 'WBC': 'wbc', 'Platelets': 'platelet'}
def convert(split):
    ids = (root/'ImageSets'/'Main'/f'{split}.txt').read_text().split()
    io = Path(f'/content/data/pbs_yolo/images/{split}'); lo = Path(f'/content/data/pbs_yolo/labels/{split}')
    io.mkdir(parents=True, exist_ok=True); lo.mkdir(parents=True, exist_ok=True)
    n = 0
    for i in ids:
        xml = root/'Annotations'/f'{i}.xml'; img = root/'JPEGImages'/f'{i}.jpg'
        if not (xml.exists() and img.exists()): continue
        t = ET.parse(xml).getroot(); sz = t.find('size')
        W = int(sz.find('width').text); H = int(sz.find('height').text)
        lines = []
        for o in t.findall('object'):
            nm = VOC2NAME.get(o.find('name').text)
            if nm is None: continue
            b = o.find('bndbox'); x1,y1,x2,y2 = (float(b.find(k).text) for k in ('xmin','ymin','xmax','ymax'))
            xc = ((x1+x2)/2)/W; yc = ((y1+y2)/2)/H; w = (x2-x1)/W; h = (y2-y1)/H
            lines.append(f'{CLASSES.index(nm)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        shutil.copy(img, io/img.name); (lo/f'{i}.txt').write_text(chr(10).join(lines)); n += 1
    return n
nt = convert('train'); nv = convert('val'); convert('test')
DATA_YAML = '/content/data/pbs_yolo/data.yaml'
open(DATA_YAML,'w').write(f'path: /content/data/pbs_yolo\ntrain: images/train\nval: images/val\nnc: {len(CLASSES)}\nnames: {CLASSES}\n')
print(f'✓ BCCD ready: {nt} train / {nv} val  ->  {DATA_YAML}')

### Option B — Roboflow abnormality set (needs a FREE key)
For RBC/WBC abnormalities. app.roboflow.com → Settings → API key. Example below is the
*Blood Detection* project (echinocyte, spherocyte, teardrop, sickle, elliptocyte, stomatocyte).

In [ ]:
# Skip if you ran Option A. Otherwise uncomment and run:
# from getpass import getpass
# import os
# os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
# from roboflow import Roboflow
# rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# ds = rf.workspace('mayukh-banerjee').project('blood-detection-puabf').version(1).download('yolov8', location='/content/data/pbs_roboflow')
# DATA_YAML = os.path.join(ds.location, 'data.yaml'); print('DATA_YAML =', DATA_YAML)
#
# TIP: to cover BOTH normal + abnormal, train Option A first, save pbs.pt, then run
# this notebook again with Option B using USE_MALARIA_BASE-style transfer from that pbs.pt.

### 🔀 Option C — merge normal + abnormal into ONE dataset (recommended for full coverage)
Combines BCCD (Option A) with a Roboflow abnormality set (Option B) into a single
unified class map, so one fine-tune run detects **normal cells AND abnormalities together**.
Run Option A, then the Option B download only, then this cell.

In [ ]:
# Option C — MERGE Option A (BCCD normal cells) + a Roboflow abnormality set into
# ONE dataset, so a SINGLE fine-tune run covers normal AND abnormal together.
# Steps: run Option A (BCCD); then in Option B uncomment ONLY the download line
# (leave DATA_YAML unset so it stays at /content/data/pbs_roboflow); then run this.
import os, glob, shutil, yaml
from pathlib import Path

CANDIDATES = ['/content/data/pbs_yolo', '/content/data/pbs_roboflow']

def find_yaml(d):
    p = os.path.join(d, 'data.yaml')
    if os.path.exists(p): return p
    hits = glob.glob(d + '/**/data.yaml', recursive=True)
    return hits[0] if hits else None

sources = [(d, find_yaml(d)) for d in CANDIDATES if find_yaml(d)]
assert sources, 'No datasets found — run Option A and/or Option B first.'

def norm(n): return str(n).strip().lower().replace(' ', '_').replace('-', '_')

unified, parsed = [], []
for d, y in sources:
    names = yaml.safe_load(open(y))['names']
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    names = [norm(n) for n in names]
    parsed.append((os.path.dirname(y), names))
    for n in names:
        if n not in unified: unified.append(n)
print('unified classes:', unified)

OUT = '/content/data/pbs_merged'
shutil.rmtree(OUT, ignore_errors=True)
for base, names in parsed:
    remap = {i: unified.index(n) for i, n in enumerate(names)}
    tag = Path(base).name
    for split in ('train', 'val', 'valid', 'test'):
        imgdirs = glob.glob(os.path.join(base, split, 'images')) + glob.glob(os.path.join(base, 'images', split))
        for imgdir in imgdirs:
            lbldir = imgdir.replace('images', 'labels')
            out = 'val' if split in ('val', 'valid') else split
            oi = Path(OUT) / 'images' / out; ol = Path(OUT) / 'labels' / out
            oi.mkdir(parents=True, exist_ok=True); ol.mkdir(parents=True, exist_ok=True)
            for img in glob.glob(os.path.join(imgdir, '*')):
                nm = f'{tag}_{os.path.basename(img)}'
                shutil.copy(img, oi / nm)
                lp = os.path.join(lbldir, Path(img).stem + '.txt')
                lines = []
                if os.path.exists(lp):
                    for ln in open(lp):
                        p = ln.split()
                        if p:
                            p[0] = str(remap.get(int(p[0]), int(p[0]))); lines.append(' '.join(p))
                (ol / (Path(nm).stem + '.txt')).write_text(chr(10).join(lines))

# carve a val split if none of the sources provided one
timg = Path(OUT) / 'images' / 'train'; vimg = Path(OUT) / 'images' / 'val'
if timg.exists() and not (vimg.exists() and any(vimg.glob('*'))):
    (Path(OUT)/'images'/'val').mkdir(parents=True, exist_ok=True)
    (Path(OUT)/'labels'/'val').mkdir(parents=True, exist_ok=True)
    for p in sorted(timg.glob('*'))[:max(1, len(list(timg.glob('*')))//7)]:
        shutil.move(str(p), str(vimg / p.name))
        lb = Path(OUT)/'labels'/'train'/f'{p.stem}.txt'
        if lb.exists(): shutil.move(str(lb), str(Path(OUT)/'labels'/'val'/lb.name))

DATA_YAML = os.path.join(OUT, 'data.yaml')
open(DATA_YAML, 'w').write(f'path: {OUT}\ntrain: images/train\nval: images/val\nnc: {len(unified)}\nnames: {unified}\n')
nt = len(list((Path(OUT)/'images'/'train').glob('*'))); nv = len(list((Path(OUT)/'images'/'val').glob('*')))
print(f'✓ merged: {nt} train / {nv} val, {len(unified)} classes  ->  {DATA_YAML}')

## 5. Choose the base weights — fine-tuning (never from scratch)
`yolov8s.pt` = COCO-pretrained (default). `malaria.pt` = your blood-microscopy model —
blood-domain transfer, usually a better starting point for smears.

In [ ]:
import os
USE_MALARIA_BASE = False   # True -> fine-tune from your malaria detector (blood-domain transfer)
malaria_pt = '/content/nexus/backend/models/malaria/malaria.pt'
if USE_MALARIA_BASE and os.path.exists(malaria_pt):
    BASE_WEIGHTS = malaria_pt
else:
    BASE_WEIGHTS = 'yolov8s.pt'   # auto-downloaded; yolov8m.pt = more accurate/slower
print('fine-tuning from:', BASE_WEIGHTS)

## 6. Fine-tune the detector
~100 epochs on a T4. Augmentation tuned for stained microscopy. Weights write to Drive
when available, so a runtime reset won't lose them.

In [ ]:
EPOCHS = 100
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
import os
PROJECT = '/content/runs/detect'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/nexus_pbs/runs'; os.makedirs(PROJECT, exist_ok=True)
    print('runs -> Drive:', PROJECT)
except Exception as e:
    print('Drive not mounted, runs stay on /content:', e)
model = YOLO(BASE_WEIGHTS)   # loads pretrained weights -> fine-tunes; head adapts to new classes
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16,
                      project=PROJECT, name='pbs', pretrained=True, patience=25, plots=True,
                      hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=180,
                      fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
m = model.val()
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 6b. RESUME instead of restart (continue from your epoch-45 checkpoint)
Your last PBS run stopped around **epoch 45/100**. Run **this cell instead of step 6**
so it picks up from the saved checkpoint rather than starting over:
- If the full run folder is on Drive (`nexus_pbs/runs/.../weights/last.pt`) it does a
  **true resume** — keeps optimizer state and continues to 100.
- Otherwise it **continues** from the newest `.pt` it finds (Drive backup, or a file you
  upload to `/content/` via the Files panel) as a fresh run to `TARGET_EPOCHS`. Run
  **step 4** first so the dataset (`DATA_YAML`) is ready for that fallback.

In [ ]:
# ── 6b. RESUME / CONTINUE PBS training — robust (no COCO head, no 4-vs-80 mismatch) ──
# Run this INSTEAD of step 6 so a disconnected run continues where it stopped.
# Fixes the `size mismatch ... [4] vs [80]` crash: the model is ALWAYS rebuilt from
# your own checkpoint (its real class count), never from an 80-class COCO base, and a
# failed true-resume falls back automatically to continue-from-weights.
import os, glob, yaml
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount:', e)
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO

DRIVE_RUNS = '/content/drive/MyDrive/nexus_pbs/runs'
# 1) A full interrupted run (has last.pt + args.yaml).
run_lasts = sorted(glob.glob(DRIVE_RUNS + '/**/weights/last.pt', recursive=True), key=os.path.getmtime)
# 2) else any loose checkpoint you downloaded/uploaded — but NEVER a plain COCO yolov8*.pt
#    (picking yolov8s.pt here is what silently starts an 80-class model over your smear model).
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_pbs/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
if not (run_lasts or loose):
    raise FileNotFoundError(
        'No checkpoint found. Upload your epoch-45 last.pt to /content/ (Files panel), '
        'or ensure Drive has /content/drive/MyDrive/nexus_pbs/... then rerun.')
ckpt = (run_lasts or loose)[-1]
print('checkpoint:', ckpt)

# Read the checkpoint's OWN classes — the model must keep exactly these.
probe = YOLO(ckpt)
ck_names = list(probe.model.names.values()) if isinstance(probe.model.names, dict) else list(probe.model.names)
print('checkpoint classes (%d): %s' % (len(ck_names), ck_names))

TARGET_EPOCHS = 100
model = None

# 1) TRUE resume (keeps optimizer state, continues to the run's original target).
#    Falls back on ANY error — the 4-vs-80 RuntimeError no longer kills the run.
if ckpt in run_lasts:
    try:
        print('trying true resume (keeps optimizer state)…')
        model = YOLO(ckpt); model.train(resume=True)
    except Exception as e:
        print('resume failed -> continuing from weights instead:', e)
        model = None

# 2) CONTINUE from weights — architecture comes FROM the checkpoint (correct class count).
if model is None:
    assert 'DATA_YAML' in globals(), 'Run step 4 (dataset) first so DATA_YAML is set.'
    d = yaml.safe_load(open(DATA_YAML)); dn = d['names']
    dn = [dn[k] for k in sorted(dn)] if isinstance(dn, dict) else list(dn)
    if [str(x).lower() for x in dn] != [str(x).lower() for x in ck_names]:
        print('⚠ dataset classes %s != checkpoint classes %s' % (dn, ck_names))
        print('  Training will re-init the head to the dataset classes (you lose the extra ones).')
        print('  To KEEP all %d classes, rebuild the SAME dataset the run used' % len(ck_names))
        print('  (Option C merged BCCD + sickle-cell set), not Option A (BCCD, 3 classes) alone.')
    print('CONTINUE (new run) from:', ckpt, '->', TARGET_EPOCHS, 'epochs')
    model = YOLO(ckpt)
    results = model.train(data=DATA_YAML, epochs=TARGET_EPOCHS, imgsz=640, batch=16,
                          project=DRIVE_RUNS, name='pbs_resume', pretrained=True,
                          patience=25, plots=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                          degrees=180, fliplr=0.5, flipud=0.5, mosaic=1.0,
                          translate=0.1, scale=0.5)

m = model.val()
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export `pbs.pt` (app path + Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/nexus_pbs'; os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    print('Drive not mounted (continuing):', e)
best_hits = [c for c in glob.glob('/content/**/weights/best.pt', recursive=True) if os.path.isfile(c)]
last_hits = [c for c in glob.glob('/content/**/weights/last.pt', recursive=True) if os.path.isfile(c)]
backup    = [c for c in glob.glob((DRIVE or '') + '/**/*.pt', recursive=True) if DRIVE and os.path.isfile(c)]
pool = best_hits or last_hits or backup
if not pool:
    raise FileNotFoundError('No trained weights found — run step 6 (training) first.')
best = sorted(pool, key=os.path.getmtime)[-1]
print('using weights:', best, '->', 'best.pt' if best_hits else ('last.pt' if last_hits else 'Drive backup'))
os.makedirs('/content/nexus/backend/models/pbs', exist_ok=True)
dest = '/content/nexus/backend/models/pbs/pbs.pt'; shutil.copy(best, dest); print('saved to app path:', dest)
if DRIVE:
    shutil.copy(best, os.path.join(DRIVE, 'pbs.pt')); print('backed up to Drive:', os.path.join(DRIVE, 'pbs.pt'))
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
try:
    YOLO(best).export(format='onnx', opset=12)
except Exception as e:
    print('onnx export skipped:', e)
from google.colab import files
files.download(best)

## 8. Put it in the app
1. Move the downloaded `pbs.pt` to **`backend/models/pbs/pbs.pt`**, commit, push (merge to `main`).
2. The vision service auto-loads it for `image_type` ∈ {`pbs`, `peripheral_blood_smear`}.
3. On Render, the local detector needs the ML deps — build with `INSTALL_ML=1` on a ≥2 GB
   instance. Without it, Claude vision still reads smears.

Detected morphologies are matched to **related disorders** via
`backend/ai_services/pbs_disorders.json` (e.g. schistocytes → MAHA; blasts → acute leukaemia).